In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path("../../data")
X_train = pd.read_csv(data_dir / "X_train.csv")
X_test = pd.read_csv(data_dir / "X_test.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").values.ravel()
y_test = pd.read_csv(data_dir / "y_test.csv").values.ravel()

print("Loaded:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Loaded:
X_train: (227972, 35)
X_test:  (10852, 35)
y_train: (227972,)
y_test:  (10852,)


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

X_all = pd.concat(
    [X_train, X_test],
    ignore_index=True,
)

y_all = np.concatenate(
    [y_train, y_test]
)

final_logistic_model = LogisticRegression(
    C=0.1,
    solver="lbfgs",
    l1_ratio=0,
    max_iter=10000,
)

final_logistic_model.fit(X_all, y_all)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",10000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'l

In [4]:
from pathlib import Path
import joblib

artifacts_dir = Path("../../artifacts")
artifacts_dir.mkdir(exist_ok=True)

joblib.dump(
    final_logistic_model,
    artifacts_dir / "logistic_model.joblib",
)

['..\\..\\artifacts\\logistic_model.joblib']

In [5]:
import json

feature_columns = list(X_all.columns)

with open(
    artifacts_dir / "feature_columns.json",
    "w",
) as f:
    json.dump(
        feature_columns,
        f,
        indent=4,
    )

In [6]:
import json
import random
from pathlib import Path

import joblib
import numpy as np

from src.inference.predict import predict_matchup
from src.inference.team_lookup import (
    load_final_team_states,
    get_team_state,
)
from src.inference.matchup_features import build_matchup_features


# ============================================================
# Setup
# ============================================================

PROJECT_ROOT = Path("../..")

MODEL_PATH = PROJECT_ROOT / "artifacts" / "logistic_model.joblib"
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "artifacts" / "feature_columns.json"

states = load_final_team_states()
model = joblib.load(MODEL_PATH)

with open(FEATURE_COLUMNS_PATH) as f:
    feature_columns = json.load(f)

TOL = 1e-10

print("=" * 60)
print("INFERENCE SANITY CHECKS")
print("=" * 60)


# ============================================================
# 1. Artifact checks
# ============================================================

assert MODEL_PATH.exists(), "Model file does not exist."
assert FEATURE_COLUMNS_PATH.exists(), "Feature-columns file does not exist."

assert len(feature_columns) > 0
assert len(feature_columns) == len(set(feature_columns)), (
    "Duplicate feature names found."
)

assert hasattr(model, "predict_proba"), (
    "Loaded model does not support predict_proba."
)

assert model.n_features_in_ == len(feature_columns), (
    f"Model expects {model.n_features_in_} features, "
    f"but feature_columns contains {len(feature_columns)}."
)

print("✓ Artifacts valid")


# ============================================================
# 2. Final-team-state table checks
# ============================================================

required_metadata = {"TeamID", "TeamName", "Season"}

assert required_metadata.issubset(states.columns), (
    "Missing required metadata columns."
)

assert not states.empty
assert states["TeamID"].notna().all()
assert states["TeamName"].notna().all()
assert states["Season"].notna().all()

# Every team-season should appear exactly once.
duplicates = states.duplicated(
    subset=["TeamID", "Season"],
    keep=False,
)

assert not duplicates.any(), (
    "Duplicate TeamID/Season rows found."
)

print("✓ Final team-state table valid")


# ============================================================
# Choose two valid team-seasons for deterministic checks
# ============================================================

team_a_name = "Duke"
team_a_season = 2003

team_b_name = "Duke"
team_b_season = 2004

team_a = get_team_state(
    states,
    team_a_name,
    team_a_season,
)

team_b = get_team_state(
    states,
    team_b_name,
    team_b_season,
)

print("✓ Team lookup works")


# ============================================================
# 3. Full matchup construction
# ============================================================

X_matchup = build_matchup_features(
    team_1_state=team_a,
    team_2_state=team_b,
    team_1_location=0,
)

assert len(X_matchup) == 1
assert X_matchup.columns.is_unique

missing_features = [
    col for col in feature_columns
    if col not in X_matchup.columns
]

assert not missing_features, (
    f"Inference matchup is missing model features: "
    f"{missing_features}"
)

X_selected = X_matchup[feature_columns]

assert list(X_selected.columns) == feature_columns
assert X_selected.shape[1] == model.n_features_in_

print("✓ Matchup feature construction valid")
print(f"  Full matchup features: {X_matchup.shape[1]}")
print(f"  Selected model features: {X_selected.shape[1]}")


# ============================================================
# 4. No NaN / infinity in selected model input
# ============================================================

values = X_selected.to_numpy(dtype=float)

assert np.isfinite(values).all(), (
    "NaN or infinity found in inference features."
)

print("✓ Model inputs contain no NaN/inf")


# ============================================================
# 5. Direct model prediction validity
# ============================================================

proba = model.predict_proba(X_selected)

assert proba.shape == (1, 2)
assert np.all(proba >= 0)
assert np.all(proba <= 1)
assert abs(proba.sum() - 1.0) < TOL

print("✓ Direct predict_proba valid")


# ============================================================
# 6. Public predict_matchup() output
# ============================================================

result = predict_matchup(
    team_1_name=team_a_name,
    team_1_season=team_a_season,
    team_2_name=team_b_name,
    team_2_season=team_b_season,
    team_1_location=0,
)

p1 = result["team_1_win_probability"]
p2 = result["team_2_win_probability"]

assert 0 <= p1 <= 1
assert 0 <= p2 <= 1
assert abs(p1 + p2 - 1.0) < TOL

# Public function should agree with direct model call.
assert abs(p1 - proba[0, 1]) < TOL
assert abs(p2 - proba[0, 0]) < TOL

print("✓ predict_matchup output valid")


# ============================================================
# 7. Neutral-team swap symmetry
# ============================================================

forward = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    0,
)

reverse = predict_matchup(
    team_b_name,
    team_b_season,
    team_a_name,
    team_a_season,
    0,
)

assert abs(
    forward["team_1_win_probability"]
    - reverse["team_2_win_probability"]
) < TOL

assert abs(
    forward["team_2_win_probability"]
    - reverse["team_1_win_probability"]
) < TOL

print("✓ Neutral swap symmetry valid")


# ============================================================
# 8. Home/away swap symmetry
# ============================================================

forward_home = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    1,
)

reverse_away = predict_matchup(
    team_b_name,
    team_b_season,
    team_a_name,
    team_a_season,
    -1,
)

assert abs(
    forward_home["team_1_win_probability"]
    - reverse_away["team_2_win_probability"]
) < TOL

assert abs(
    forward_home["team_2_win_probability"]
    - reverse_away["team_1_win_probability"]
) < TOL

print("✓ Home/away swap symmetry valid")


# ============================================================
# 9. Same team-season on neutral court
# ============================================================

same_team = predict_matchup(
    team_a_name,
    team_a_season,
    team_a_name,
    team_a_season,
    0,
)

same_p = same_team["team_1_win_probability"]

print(
    f"  Same-team neutral probability: "
    f"{same_p:.6f}"
)

# With mirrored training this should be essentially 0.5.
assert abs(same_p - 0.5) < 1e-6, (
    "Same team-season on neutral court is not ~50/50."
)

print("✓ Same-team neutral check valid")


# ============================================================
# 10. Location should actually affect predictions
# ============================================================

neutral = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    0,
)["team_1_win_probability"]

home = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    1,
)["team_1_win_probability"]

away = predict_matchup(
    team_a_name,
    team_a_season,
    team_b_name,
    team_b_season,
    -1,
)["team_1_win_probability"]

print(
    f"  Away={away:.4f}, "
    f"Neutral={neutral:.4f}, "
    f"Home={home:.4f}"
)

assert home > neutral > away, (
    "Expected home > neutral > away probability."
)

print("✓ Location behaves sensibly")


# ============================================================
# 11. Invalid location rejected
# ============================================================

try:
    predict_matchup(
        team_a_name,
        team_a_season,
        team_b_name,
        team_b_season,
        7,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Invalid team_1_location was not rejected."
    )

print("✓ Invalid location rejected")


# ============================================================
# 12. Invalid team-season rejected
# ============================================================

try:
    predict_matchup(
        team_1_name="Definitely Not A Team",
        team_1_season=9999,
        team_2_name=team_b_name,
        team_2_season=team_b_season,
        team_1_location=0,
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "Invalid team-season was not rejected."
    )

print("✓ Invalid team-season rejected")


# ============================================================
# 13. Random batch smoke test
# ============================================================

records = states[
    ["TeamName", "Season"]
].drop_duplicates().to_records(index=False)

random.seed(42)

for _ in range(100):
    a, b = random.sample(list(records), 2)

    r = predict_matchup(
        team_1_name=str(a[0]),
        team_1_season=int(a[1]),
        team_2_name=str(b[0]),
        team_2_season=int(b[1]),
        team_1_location=random.choice([-1, 0, 1]),
    )

    p1 = r["team_1_win_probability"]
    p2 = r["team_2_win_probability"]

    assert np.isfinite(p1)
    assert np.isfinite(p2)
    assert 0 <= p1 <= 1
    assert 0 <= p2 <= 1
    assert abs(p1 + p2 - 1.0) < TOL

print("✓ 100 random matchup predictions passed")


# ============================================================
# Done
# ============================================================

print()
print("=" * 60)
print("ALL INFERENCE SANITY CHECKS PASSED")
print("=" * 60)

INFERENCE SANITY CHECKS
✓ Artifacts valid
✓ Final team-state table valid
✓ Team lookup works


AssertionError: Inference matchup is missing model features: ['away_strength_diff', 'wins_ratio_diff', 'home_strength_diff']

Project: NCAA Historical Matchup Predictor Web App

Goal:
Build a simple web app where a user selects:
- Team 1
- Team 1 season
- Team 2
- Team 2 season
- Team 1 location: Home / Neutral / Away

The app should call the existing inference backend and display both teams' model-estimated win probabilities.

Important: the ML/backend inference is already finished. Do not retrain models or recreate feature engineering in the web app.

Backend assets already available:

1. data/final_team_states.csv
   - One row per team-season.
   - Contains:
     TeamID
     TeamName
     Season
     full end-of-regular-season feature state
   - This file should also be used to populate valid team/season dropdowns.

2. artifacts/logistic_model.joblib
   - Fully fitted final LogisticRegression model.
   - Trained on all available labeled data after model evaluation was completed.

3. artifacts/feature_columns.json
   - Exact selected feature names, in exact order, expected by the saved logistic regression.
   - Current deployed model uses 35 selected features.

Existing inference package:

src/inference/
    team_lookup.py
    matchup_features.py
    predict.py

Public backend function:

predict_matchup(
    team_1_name: str,
    team_1_season: int,
    team_2_name: str,
    team_2_season: int,
    team_1_location: int = 0,
) -> dict

Location coding:
    1  = Team 1 home
    0  = neutral
   -1  = Team 1 away

Example:

result = predict_matchup(
    team_1_name="Duke",
    team_1_season=2003,
    team_2_name="Duke",
    team_2_season=2004,
    team_1_location=0,
)

Expected result structure:

{
    "team_1": "Duke 2003",
    "team_2": "Duke 2004",
    "team_1_win_probability": 0.4477,
    "team_2_win_probability": 0.5523
}

The inference flow already works as follows:

saved team-season states
    ↓
lookup Team 1 + Team 2
    ↓
construct full differential matchup row
    ↓
select exact saved feature_columns
    ↓
saved logistic regression
    ↓
predict_proba()
    ↓
return both probabilities

This inference layer has already passed sanity tests:
- probabilities sum to 1
- neutral team-order reversal gives complementary probabilities
- home/away reversal gives complementary probabilities
- identical team-season vs itself on neutral gives exactly ~50/50
- location affects predictions sensibly
- invalid team-season combinations are rejected
- invalid locations are rejected
- 100 random matchup predictions completed successfully

Web-app requirements:

Use final_team_states.csv as the source of truth for valid selections.

The UI should prevent invalid team-season combinations. For example:
- after choosing a team, only show seasons that exist for that team
OR
- after choosing a season, only show teams that exist in that season

Inputs:
- Team 1 dropdown
- Team 1 season dropdown
- Team 2 dropdown
- Team 2 season dropdown
- Location selector:
    Team 1 Home
    Neutral
    Team 1 Away
- Predict button

On click:
- call predict_matchup(...)
- display both probabilities prominently
- optionally highlight the predicted winner

Example display:

Duke 2003        63.2%
Duke 2004        36.8%

Predicted winner: Duke 2003

Important architectural constraint:
The web app should be a thin presentation layer. It should import and call src.inference.predict.predict_matchup rather than duplicating:
- team lookup
- differential feature construction
- feature filtering
- model loading logic
- ML training

The deployed app only needs:
- app code
- src/inference/
- data/final_team_states.csv
- artifacts/logistic_model.joblib
- artifacts/feature_columns.json
- Python dependencies

Training notebooks, raw NCAA source files, X_train/X_test, feature-selection code, and other experimental models are not required at runtime.

Recommended MVP:
Streamlit is perfectly suitable because this is primarily dropdown selection + prediction output.

Main objective:
Create a clean, polished historical "what-if" NCAA matchup experience, e.g.

Duke 2019 vs Duke 2024
UConn 2023 vs Kentucky 2012
etc.

The displayed percentages should be described as "model-estimated win probability," since these are hypothetical cross-season matchups.